# 参数与维表全量数据质量审计

## tl;dr

- 未发现第二个与“单媒体智投点击误用曝光参数”同等级的已确认行为映射错误。
- 17 个组件、56 条参数定义、63 条行为映射、67 条场景映射均通过结构完整性、业务键唯一性和复合输出键唯一性检查。
- 发现 3 类维护风险：单媒体相对日期模板与官方输出合同不一致（运行时被特殊逻辑修正）、CSV 与已发布数据库存在有意/历史漂移、一个品牌别名输出值重复。
- `全媒体智投` 曝光/点击共用 Value 不是反置：最终 `bhv` 的 ID 不同，并由 `bhv_type` 区分，现有官方回归样例明确覆盖。

## Context & Methods

### Key Assumptions

- 当前生产样式的运行来源是 `.runtime/cdp.db` 中已发布且启用的维表记录；CSV 是初始种子与代码仓库基线。
- 行为、场景、状态和渠道的有效标识是复合值（如 `ID#|#Value`），不能仅凭 ID 或 Value 单列重复判错。
- 单媒体智投与品牌推广的对照合同来自用户在本任务中提供的官方 JSON；其他广告组件以项目中标为 official/supplied 的回归样例作为内部依据。
- 本审计只读配置和数据库，不发布、不修改维表。

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import json
import sqlite3
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "1.参数表.csv").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DB_PATH = ROOT / ".runtime" / "cdp.db"
PARAM_PATH = ROOT / "1.参数表.csv"

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def read_csv_flexible(path):
    for encoding in ("utf-8-sig", "gb18030"):
        try:
            with path.open("r", encoding=encoding, newline="") as handle:
                return list(csv.DictReader(handle)), encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"Unable to decode {path}")

def normalized_record(row):
    return tuple(sorted((str(key).strip(), str(value or "").strip()) for key, value in row.items()))

assert DB_PATH.exists(), DB_PATH
assert PARAM_PATH.exists(), PARAM_PATH
print(f"Workspace: {ROOT}")
print(f"Database: {DB_PATH}")

Workspace: E:\CDP_Project_codex
Database: E:\CDP_Project_codex\.runtime\cdp.db


## Data

读取根目录参数表、17 张逻辑表、9 张当前管理维表及一张未接入的功效维表，并读取配置版本 9 的已发布数据库快照。

In [2]:
params, params_encoding = read_csv_flexible(PARAM_PATH)
logic_files = sorted(ROOT.glob("*逻辑表*.csv"))
dimension_files = sorted(ROOT.glob("*维表.csv"))

conn = sqlite3.connect(f"file:{DB_PATH.as_posix()}?mode=ro", uri=True)
conn.row_factory = sqlite3.Row
latest = conn.execute("SELECT version_number, note, published_at FROM config_versions ORDER BY version_number DESC LIMIT 1").fetchone()

dataset_rows = []
for path in [PARAM_PATH, *logic_files, *dimension_files]:
    rows, encoding = read_csv_flexible(path)
    dataset_rows.append({"文件": path.name, "行数": len(rows), "字段数": len(rows[0]) if rows else 0, "编码": encoding})

dataset_summary = pd.DataFrame(dataset_rows)
print(f"参数组件数: {len({row['Crowd_Package'] for row in params})}")
print(f"参数定义数: {len(params)}")
print(f"逻辑表数: {len(logic_files)}")
print(f"维表数（含未接入功效维表）: {len(dimension_files)}")
print(f"已发布配置版本: {latest['version_number']}，说明: {latest['note']}")
dataset_summary

参数组件数: 17
参数定义数: 56
逻辑表数: 17
维表数（含未接入功效维表）: 10
已发布配置版本: 9，说明: 修正单媒体智投点击行为参数


,文件,行数,字段数,编码
0,1.参数表.csv,56,12,utf-8-sig
1,10.快消策略人群3.0_逻辑表.csv,1,2,gb18030
2,11.预测性别_逻辑表_逻辑表.csv,1,2,gb18030
3,12.月均消费金额_逻辑表.csv,1,2,gb18030
4,13.关键词搜索_逻辑表.csv,1,3,utf-8-sig
5,14.品牌专区_逻辑表.csv,2,4,utf-8-sig
6,15.效果推广_逻辑表.csv,3,5,utf-8-sig
7,16.全媒体智投_逻辑表.csv,2,3,utf-8-sig
8,17.单媒体智投_逻辑表.csv,2,2,utf-8-sig
9,18.品牌推广_逻辑表.csv,2,5,utf-8-sig


## Results

### 1. 参数表与逻辑表覆盖

检查组件集合、字段键/标签唯一性、JSON 字段可解析性、数据源文件存在性，以及同一组件 Base Template 是否一致。

In [3]:
import re

def package_from_logic(filename):
    stem = re.sub(r"^\d+\.", "", filename)
    return stem.replace("_逻辑表_逻辑表.csv", "").replace("_逻辑表.csv", "")

parameter_issues = []
params_by_package = defaultdict(list)
for line_number, row in enumerate(params, start=2):
    package = row["Crowd_Package"].strip()
    params_by_package[package].append((line_number, row))
    for column in ("Base_Template", "Backend_Template", "UI_Config"):
        raw = (row.get(column) or "").strip()
        if raw and raw != "-":
            try:
                json.loads(raw)
            except Exception as exc:
                parameter_issues.append({"类型": "JSON无法解析", "组件": package, "行": line_number, "字段": column, "详情": str(exc)})
    source = (row.get("Data_Source") or "").strip()
    if source and source != "-" and not (ROOT / source).exists():
        parameter_issues.append({"类型": "数据源缺失", "组件": package, "行": line_number, "字段": source, "详情": "文件不存在"})

for package, items in params_by_package.items():
    for column in ("Param_Key", "Label"):
        counts = Counter(row[column].strip() for _, row in items)
        for value, count in counts.items():
            if not value or count > 1:
                parameter_issues.append({"类型": "字段为空或重复", "组件": package, "行": "-", "字段": column, "详情": f"{value!r} × {count}"})
    if len({row["Base_Template"].strip() for _, row in items}) != 1:
        parameter_issues.append({"类型": "Base Template不一致", "组件": package, "行": "-", "字段": "Base_Template", "详情": "同组件存在多个基础模板"})

logic_packages = {package_from_logic(path.name) for path in logic_files}
parameter_packages = set(params_by_package)
coverage = pd.DataFrame([
    {"检查": "参数组件都有逻辑表", "结果": not (parameter_packages - logic_packages), "差异": sorted(parameter_packages - logic_packages)},
    {"检查": "逻辑表都有参数组件", "结果": not (logic_packages - parameter_packages), "差异": sorted(logic_packages - parameter_packages)},
    {"检查": "参数 JSON、键、标签、数据源、基础模板", "结果": not parameter_issues, "差异": parameter_issues},
])
coverage

,检查,结果,差异
0,参数组件都有逻辑表,True,[]
1,逻辑表都有参数组件,True,[]
2,参数 JSON、键、标签、数据源、基础模板,True,[]


### 2. 维表完整性与输出键唯一性

按系统真实输出规则检查复合键，避免把合法的 ID/Value 单列复用误报为冲突。

In [4]:
name_columns = {
    "行为维表.csv": "行为名称", "场景维表.csv": "场景名称", "类目维表.csv": "类目名称",
    "品牌维表.csv": "品牌名称", "渠道维表.csv": "渠道名称", "账号维表.csv": "账号名称",
    "商品类型维表.csv": "类型名称", "状态维表.csv": "状态名称", "属性值维表.csv": "属性值名称",
}

def natural_key(filename, row):
    parts = [row.get("适用的包", "").strip(), row.get(name_columns[filename], "").strip()]
    if filename == "行为维表.csv":
        parts.append(row.get("适用的渠道", "").strip() or "ALL")
    elif filename == "场景维表.csv":
        parts.append(row.get("适用的行为", "").strip())
    return tuple(parts)

def output_scope_and_value(filename, row):
    package = row.get("适用的包", "").strip()
    if filename == "行为维表.csv":
        return (package, row.get("适用的渠道", "").strip() or "ALL"), f"{row.get('ID','')}#|#{row.get('Value','')}"
    if filename == "场景维表.csv":
        return (package, row.get("适用的行为", "").strip()), f"{row.get('ID','')}#|#{row.get('Value','')}"
    if filename == "状态维表.csv":
        return (package,), f"{row.get('ID','')}#|#{row.get('Value','')}"
    if filename == "渠道维表.csv":
        return (package,), f"{row.get('parentId','')}#|#{row.get('BizID','')}"
    if filename == "类目维表.csv":
        return (package,), f"{row.get('cateId','')}#|#{row.get('cateId','')}"
    if filename == "品牌维表.csv":
        return (package,), row.get("Value", "").strip()
    if filename in ("账号维表.csv", "商品类型维表.csv", "属性值维表.csv"):
        return (package,), row.get("ID", "").strip()
    raise KeyError(filename)

dimension_checks = []
output_collisions = []
for filename, name_column in name_columns.items():
    rows, _ = read_csv_flexible(ROOT / filename)
    exact_counts = Counter(normalized_record(row) for row in rows)
    natural_counts = Counter(natural_key(filename, row) for row in rows)
    grouped_outputs = defaultdict(set)
    for row in rows:
        scope, output_value = output_scope_and_value(filename, row)
        grouped_outputs[(scope, output_value)].add(row[name_column].strip())
    collisions = [
        {"文件": filename, "范围": scope, "输出值": value, "名称": sorted(names)}
        for (scope, value), names in grouped_outputs.items() if len(names) > 1
    ]
    output_collisions.extend(collisions)
    dimension_checks.append({
        "维表": filename, "行数": len(rows),
        "完全重复行": sum(count - 1 for count in exact_counts.values() if count > 1),
        "业务键重复": sum(count - 1 for count in natural_counts.values() if count > 1),
        "复合输出冲突": len(collisions),
    })

pd.DataFrame(dimension_checks)

,维表,行数,完全重复行,业务键重复,复合输出冲突
0,行为维表.csv,63,0,0,0
1,场景维表.csv,67,0,0,0
2,类目维表.csv,575,0,0,0
3,品牌维表.csv,16025,0,0,1
4,渠道维表.csv,11,0,0,0
5,账号维表.csv,8,0,0,0
6,商品类型维表.csv,2,0,0,0
7,状态维表.csv,4,0,0,0
8,属性值维表.csv,56,0,0,0


In [5]:
pd.DataFrame(output_collisions)

,文件,范围,输出值,名称
0,品牌维表.csv,"(类目公域行为,)",236210576,"[DU'IT, DU’IT]"


品牌表中的唯一输出冲突是 `DU'IT` 与 `DU’IT`（直引号/弯引号）共同映射到 `236210576`。二者语义相同，属于别名重复，不会把两个不同业务行为混淆。

### 3. 行为与场景的“单列重复”判定

单列复用仅作为人工复核线索；最终是否冲突以复合输出值为准。

In [6]:
def single_column_reuse(filename, scope_columns, id_column, name_column):
    rows, _ = read_csv_flexible(ROOT / filename)
    groups = defaultdict(set)
    for row in rows:
        key = tuple(row.get(column, "").strip() for column in [*scope_columns, id_column])
        groups[key].add(row[name_column].strip())
    return [
        {"文件": filename, "字段": id_column, "范围和值": key, "名称": sorted(names)}
        for key, names in groups.items() if len(names) > 1
    ]

reuse_candidates = []
reuse_candidates += single_column_reuse("行为维表.csv", ["适用的包"], "Value", "行为名称")
reuse_candidates += single_column_reuse("场景维表.csv", ["适用的包", "适用的行为"], "ID", "场景名称")
reuse_candidates += single_column_reuse("场景维表.csv", ["适用的包", "适用的行为"], "Value", "场景名称")
pd.DataFrame(reuse_candidates)

,文件,字段,范围和值,名称
0,行为维表.csv,Value,"(全媒体智投, EXP_UD_ZHT_EXP_BHV)","[曝光, 点击]"
1,场景维表.csv,ID,"(效果推广, 点击, 15402)","[关键词推广(原淘内广告/直通车), 原万相台-电商场景]"
2,场景维表.csv,Value,"(品牌推广, 点击, 77)","[淘内展示营销 - 竞争卡位, 集团内投]"
3,场景维表.csv,Value,"(品牌推广, 曝光, 77)","[淘内展示营销 - 竞争卡位, 集团内投]"


判定：

- 全媒体智投曝光/点击共用 `EXP_UD_ZHT_EXP_BHV`，但 ID 分别为 `19873`、`19874`，且最终还有 `bhv_type=exp_udzht/click_udzht`，属于官方合同中的合法复用。
- 效果推广点击的场景 ID `15402` 分别与 Value `371`、`-999` 组合；品牌推广 Value `77` 也由不同 ID 区分。复合输出均唯一，不构成映射冲突。

### 4. 用户提供的官方 JSON 合同

同时核对 CSV 与当前已发布数据库。

In [7]:
published_behavior_rows = {}
for db_row in conn.execute("SELECT published_data FROM dimension_rows WHERE dimension_file='行为维表.csv' AND is_published=1 AND published_enabled=1 AND published_deleted=0"):
    row = json.loads(db_row["published_data"])
    published_behavior_rows[(row["适用的包"], row["行为名称"])] = row

csv_behavior_rows, _ = read_csv_flexible(ROOT / "行为维表.csv")
csv_behavior_map = {(row["适用的包"], row["行为名称"]): row for row in csv_behavior_rows}
expected_behavior = {
    ("单媒体智投", "曝光"): ("19937", "is_pv_uddmt"),
    ("单媒体智投", "点击"): ("19938", "is_click_uddmt"),
    ("品牌推广", "曝光"): ("15318", "exp_pptg"),
    ("品牌推广", "点击"): ("15298", "click_pptg"),
}
contract_rows = []
for key, expected in expected_behavior.items():
    csv_row = csv_behavior_map[key]
    db_row = published_behavior_rows[key]
    contract_rows.append({
        "组件": key[0], "行为": key[1], "期望": "#|#".join(expected),
        "CSV": f"{csv_row['ID']}#|#{csv_row['Value']}",
        "已发布DB": f"{db_row['ID']}#|#{db_row['Value']}",
        "通过": (csv_row["ID"], csv_row["Value"]) == expected and (db_row["ID"], db_row["Value"]) == expected,
    })
pd.DataFrame(contract_rows)

,组件,行为,期望,CSV,已发布DB,通过
0,单媒体智投,曝光,19937#|#is_pv_uddmt,19937#|#is_pv_uddmt,19937#|#is_pv_uddmt,True
1,单媒体智投,点击,19938#|#is_click_uddmt,19938#|#is_click_uddmt,19938#|#is_click_uddmt,True
2,品牌推广,曝光,15318#|#exp_pptg,15318#|#exp_pptg,15318#|#exp_pptg,True
3,品牌推广,点击,15298#|#click_pptg,15298#|#click_pptg,15298#|#click_pptg,True


### 5. 单媒体相对日期模板漂移

官方合同要求相对日期只输出 `dateType=RELATIVE_RANGE`；参数表模板仍包含 `dateValue`，目前由后端单媒体专用规范化逻辑删除。运行结果正确，但配置文件本身不是唯一真源。

In [8]:
single_time = next(row for row in params if row["Crowd_Package"] == "单媒体智投" and row["Param_Key"] == "time")
single_time_rules = json.loads(single_time["Backend_Template"])
recent_template = next(rule["template"] for rule in single_time_rules if rule["trigger"] == "recent")
pd.DataFrame([{
    "检查": "单媒体相对日期模板不应含 dateValue",
    "参数表模板字段": sorted(recent_template),
    "运行时官方输出字段": ["dateType"],
    "通过": set(recent_template) == {"dateType"},
    "运行影响": "当前被 ConfigEngine._canonicalize_single_media 修正，最终 JSON 正确",
}])

,检查,参数表模板字段,运行时官方输出字段,通过,运行影响
0,单媒体相对日期模板不应含 dateValue,"[dateType, dateValue]",[dateType],False,当前被 ConfigEngine._canonicalize_single_media 修正...


### 6. CSV 与已发布数据库漂移

这些差异不等同于错误：删除品牌/渠道以及新增类目/账号都有配置审计记录。但它们说明仓库 CSV 不是已发布配置的完整灾备副本。

In [9]:
drift_rows = []
for filename in name_columns:
    csv_rows, _ = read_csv_flexible(ROOT / filename)
    db_rows = [json.loads(row["published_data"]) for row in conn.execute(
        "SELECT published_data FROM dimension_rows WHERE dimension_file=? AND is_published=1 AND published_enabled=1 AND published_deleted=0",
        (filename,),
    )]
    csv_counter = Counter(normalized_record(row) for row in csv_rows)
    db_counter = Counter(normalized_record(row) for row in db_rows)
    csv_only = list((csv_counter - db_counter).elements())
    db_only = list((db_counter - csv_counter).elements())
    if csv_only or db_only:
        drift_rows.append({"维表": filename, "CSV有效行": len(csv_rows), "DB已发布启用行": len(db_rows), "仅CSV": len(csv_only), "仅DB": len(db_only)})
pd.DataFrame(drift_rows)

,维表,CSV有效行,DB已发布启用行,仅CSV,仅DB
0,类目维表.csv,575,686,0,111
1,品牌维表.csv,16025,16024,1,0
2,渠道维表.csv,11,10,1,0
3,账号维表.csv,8,9,0,1


### 7. 运行时元数据规则

验证默认值必须属于选项、动态场景覆盖所有行为、选项无重复、引用维表不为空。

In [10]:
from cdp_backend.engine import ConfigEngine
engine = ConfigEngine(db_path=str(DB_PATH))
runtime_issues = []
for package in sorted(engine.packages):
    schema = engine.get_package_meta(package).get("schema", [])
    for field in schema:
        options = field.get("options") or []
        ui = field.get("uiConfig") or {}
        if len(options) != len(set(options)):
            runtime_issues.append((package, field.get("key"), "选项重复"))
        if ui.get("defaultValue") is not None and ui.get("defaultValue") not in options:
            runtime_issues.append((package, field.get("key"), "默认值不在选项中"))
        options_by_value = field.get("optionsByValue") or {}
        if options_by_value:
            source_key = ui.get("optionSourceKey")
            source_field = next((item for item in schema if item.get("key") == source_key), None)
            source_options = source_field.get("options", []) if source_field else []
            missing = [value for value in source_options if value not in options_by_value]
            if missing:
                runtime_issues.append((package, field.get("key"), f"缺少行为场景: {missing}"))

pd.DataFrame(runtime_issues, columns=["组件", "字段", "问题"]) if runtime_issues else pd.DataFrame([{"检查": "17个组件运行时元数据", "结果": "通过", "问题数": 0}])

,检查,结果,问题数
0,17个组件运行时元数据,通过,0


## Takeaways

1. **没有发现第二个已确认的行为 ID/Value 反置或复制错误。** 单媒体智投与品牌推广的四组用户提供合同均同时匹配 CSV 和已发布数据库。
2. **应优先消除单媒体时间模板与运行逻辑的双重真源。** 建议把参数表的相对日期模板改成只含 `dateType`，并保留后端防御性规范化。
3. **应把已发布维表导出纳入版本化备份。** 当前数据库比 CSV 多 111 个类目和 1 个商品账号，同时少 1 个已删除品牌和 1 个已删除渠道；运行正确，但仅靠仓库 CSV 无法完整重建当前选项集。
4. **品牌别名可做去重。** `DU'IT` 与 `DU’IT` 输出相同编码；这是低风险 UI 重复，不是跨业务映射。
5. **建议增加稳定自动化检查。** 至少包括业务键唯一、复合输出唯一、UI 默认值有效、动态场景覆盖、参数 JSON 可解析、用户确认合同回归，以及 CSV/DB 漂移清单。